# Apriori Algorithm for Association Rule Mining

**Objective:** Apply the Apriori algorithm on transaction data to find frequent itemsets and association rules.

**Dataset:** Market basket transaction dataset

This notebook is Colab-ready and saves tables, metrics, and visual outputs under
`results/`. Public datasets or compact sample datasets are used so the workflow
remains reproducible.


In [ ]:
!pip install -q pandas mlxtend matplotlib seaborn


In [ ]:
from pathlib import Path

import matplotlib.pyplot as plt
import pandas as pd
import seaborn as sns
from mlxtend.frequent_patterns import apriori, association_rules
from mlxtend.preprocessing import TransactionEncoder

RESULTS_DIR = Path("results")
RESULTS_DIR.mkdir(exist_ok=True)


In [ ]:
transactions = [
    ["milk", "bread", "butter"],
    ["bread", "diaper", "beer", "eggs"],
    ["milk", "diaper", "beer", "cola"],
    ["bread", "milk", "diaper", "beer"],
    ["bread", "milk", "diaper", "cola"],
    ["eggs", "milk", "bread"],
    ["bread", "butter"],
    ["milk", "bread", "diaper"],
] * 20

encoder = TransactionEncoder()
basket = pd.DataFrame(encoder.fit(transactions).transform(transactions), columns=encoder.columns_)
itemsets = apriori(basket, min_support=0.2, use_colnames=True)
rules = association_rules(itemsets, metric="confidence", min_threshold=0.45)
rules = rules.sort_values(["lift", "confidence"], ascending=False)
itemsets.to_csv(RESULTS_DIR / "frequent_itemsets.csv", index=False)
rules.to_csv(RESULTS_DIR / "association_rules.csv", index=False)
display(rules.head())


In [ ]:
top_rules = rules.head(8).copy()
top_rules["rule"] = top_rules["antecedents"].astype(str) + " -> " + top_rules["consequents"].astype(str)
fig, axes = plt.subplots(1, 2, figsize=(13, 5))
sns.barplot(data=top_rules, y="rule", x="lift", ax=axes[0])
axes[0].set_title("Top Rules by Lift")
sns.scatterplot(data=rules, x="confidence", y="lift", size="support", ax=axes[1])
axes[1].set_title("Rule Quality")
plt.tight_layout()
plt.savefig(RESULTS_DIR / "apriori_dashboard.png", dpi=180)
plt.show()
